In [2]:
import pandas as pd
from pathlib import Path

project_dir = Path.home() / "Documents" / "banking-customer-data-pipeline"

data = {
    "customer_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 109],
    "customer_name": [
        "Amit", "Rahul", "Priya", "Rohit", None,
        "Neha", "Arjun", "Sneha", "Vikram", "Vikram"
    ],
    "kyc_status": [
        "Completed", "Pending", "Completed", "Rejected", "Completed",
        "Unknown", "Completed", "Pending", "Completed", "Completed"
    ],
    "balance": [
        50000, 75000, 120000, 30000, 45000,
        -5000, 80000, 60000, 95000, 95000
    ]
}

customers = pd.DataFrame(data)

customers

,customer_id,customer_name,kyc_status,balance
0,101,Amit,Completed,50000
1,102,Rahul,Pending,75000
2,103,Priya,Completed,120000
3,104,Rohit,Rejected,30000
4,105,NaN,Completed,45000
5,106,Neha,Unknown,-5000
6,107,Arjun,Completed,80000
7,108,Sneha,Pending,60000
8,109,Vikram,Completed,95000
9,109,Vikram,Completed,95000


In [3]:
source_file = project_dir / "data" / "source" / "customers.csv"

customers.to_csv(source_file, index=False)

print("Source file saved:", source_file)

Source file saved: C:\Users\DEEPAK\Documents\banking-customer-data-pipeline\data\source\customers.csv


In [4]:
source_customers = pd.read_csv(source_file)

print("Number of source records is:", len(source_customers))
source_customers.head()

Number of source records is: 10


,customer_id,customer_name,kyc_status,balance
0,101,Amit,Completed,50000
1,102,Rahul,Pending,75000
2,103,Priya,Completed,120000
3,104,Rohit,Rejected,30000
4,105,NaN,Completed,45000


In [5]:
source_customers[source_customers['customer_name'].isna()]

,customer_id,customer_name,kyc_status,balance
4,105,NaN,Completed,45000


In [6]:
allowed_status = ["Completed", "Pending", "Rejected"]
source_customers[~source_customers["kyc_status"].isin(allowed_status)]

,customer_id,customer_name,kyc_status,balance
5,106,Neha,Unknown,-5000


In [7]:
 source_customers[source_customers["customer_id"].duplicated(keep=False)]

,customer_id,customer_name,kyc_status,balance
8,109,Vikram,Completed,95000
9,109,Vikram,Completed,95000


In [6]:
source_customers[source_customers["balance"] < 0]

,customer_id,customer_name,kyc_status,balance
5,106,Neha,Unknown,-5000


In [8]:
allowed_status = ["Completed", "Pending", "Rejected"]

source_customers["missing_name"] = (
    source_customers["customer_name"].isna()
)

source_customers["invalid_kyc"] = (
    ~source_customers["kyc_status"].isin(allowed_status)
)

source_customers["negative_balance"] = (
    source_customers["balance"] < 0
)

source_customers["duplicate_customer_id"] = (
    source_customers["customer_id"].duplicated(keep=False)
)

source_customers

,customer_id,customer_name,kyc_status,balance,missing_name,invalid_kyc,negative_balance,duplicate_customer_id
0,101,Amit,Completed,50000,False,False,False,False
1,102,Rahul,Pending,75000,False,False,False,False
2,103,Priya,Completed,120000,False,False,False,False
3,104,Rohit,Rejected,30000,False,False,False,False
4,105,NaN,Completed,45000,True,False,False,False
5,106,Neha,Unknown,-5000,False,True,True,False
6,107,Arjun,Completed,80000,False,False,False,False
7,108,Sneha,Pending,60000,False,False,False,False
8,109,Vikram,Completed,95000,False,False,False,True
9,109,Vikram,Completed,95000,False,False,False,True


In [9]:
# List the validation columns
validation_columns = [
    "missing_name",
    "invalid_kyc",
    "negative_balance",
    "duplicate_customer_id"
]

# Identify rows that fail at least one validation rule
failed_rows = source_customers[
    source_customers[validation_columns].any(axis=1)
].copy()

failed_rows

,customer_id,customer_name,kyc_status,balance,missing_name,invalid_kyc,negative_balance,duplicate_customer_id
4,105,NaN,Completed,45000,True,False,False,False
5,106,Neha,Unknown,-5000,False,True,True,False
8,109,Vikram,Completed,95000,False,False,False,True
9,109,Vikram,Completed,95000,False,False,False,True


In [10]:
def get_failure_reasons(row):
    reasons = []

    if row["missing_name"]:
        reasons.append("Missing customer name")

    if row["invalid_kyc"]:
        reasons.append("Invalid KYC status")

    if row["negative_balance"]:
        reasons.append("Negative balance")

    if row["duplicate_customer_id"]:
        reasons.append("Duplicate customer ID")

    return "; ".join(reasons)


failed_rows["failure_reasons"] = failed_rows.apply(
    get_failure_reasons, axis=1
)

failed_rows[
    ["customer_id", "customer_name", "failure_reasons"]
]

,customer_id,customer_name,failure_reasons
4,105,NaN,Missing customer name
5,106,Neha,Invalid KYC status; Negative balance
8,109,Vikram,Duplicate customer ID
9,109,Vikram,Duplicate customer ID


In [11]:
validation_columns = [
    "missing_name",
    "invalid_kyc",
    "negative_balance",
    "duplicate_customer_id"
]

# Records that pass all validation rules
valid_customers = source_customers[
    ~source_customers[validation_columns].any(axis=1)
].copy()

# Records that fail at least one rule
exception_customers = source_customers[
    source_customers[validation_columns].any(axis=1)
].copy()

print("Valid records:", len(valid_customers))
print("Exception records:", len(exception_customers))

Valid records: 6
Exception records: 4


In [12]:
valid_customers.to_csv(
    project_dir / "data" / "source" / "valid_customers.csv",
    index=False
)

exception_customers.to_csv(
    project_dir / "data" / "exceptions" / "exception_customers.csv",
    index=False
)

print("Both files saved successfully!")

Both files saved successfully!


In [13]:
source_count = len(source_customers)
valid_count = len(valid_customers)
exception_count = len(exception_customers)

print("Source count:", source_count)
print("Valid count:", valid_count)
print("Exception count:", exception_count)
print("Reconciled count:", valid_count + exception_count)

if source_count == valid_count + exception_count:
    print("RECONCILIATION PASSED")
else:
    print("RECONCILIATION FAILED")

Source count: 10
Valid count: 6
Exception count: 4
Reconciled count: 10
RECONCILIATION PASSED


In [19]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv(project_dir / ".env")

db_config = {
    "host": os.getenv("DB_HOST"),
    "port": int(os.getenv("DB_PORT")),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "database": os.getenv("DB_NAME")
}

connection = mysql.connector.connect(**db_config)
cursor = connection.cursor()

insert_query = """
INSERT INTO customer_master
(customer_id, customer_name, kyc_status, balance)
VALUES (%s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    customer_name = VALUES(customer_name),
    kyc_status = VALUES(kyc_status),
    balance = VALUES(balance)
"""

records = list(
    valid_customers[
        ["customer_id", "customer_name", "kyc_status", "balance"]
    ].itertuples(index=False, name=None)
)

cursor.executemany(insert_query, records)
connection.commit()

print(cursor.rowcount, "records inserted successfully")

cursor.close()
connection.close()

2 records inserted successfully


In [21]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv(project_dir / ".env")

db_config = {
    "host": os.getenv("DB_HOST"),
    "port": int(os.getenv("DB_PORT")),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "database": os.getenv("DB_NAME")
}

connection = mysql.connector.connect(**db_config)
cursor = connection.cursor(dictionary=True)

cursor.execute("SELECT * FROM customer_master")

rows = cursor.fetchall()

target_df = pd.DataFrame(rows)

print(target_df)

   customer_id customer_name kyc_status    balance
0          101          Amit  Completed   50000.00
1          102         Rahul    Pending   75000.00
2          103         Priya  Completed  120000.00
3          104         Rohit   Rejected   30000.00
4          107         Arjun  Completed   80000.00
5          108         Sneha    Pending   60000.00


In [22]:
source_ids = set(valid_customers["customer_id"])
target_ids = set(target_df["customer_id"])

print("Missing in target:", source_ids - target_ids)
print("Unexpected in target:", target_ids - source_ids)

print("Source valid count:", len(valid_customers))
print("Target count:", len(target_df))

Missing in target: set()
Unexpected in target: set()
Source valid count: 6
Target count: 6


In [23]:
cursor.close()
connection.close()

In [24]:
comparison = valid_customers.merge(
    target_df,
    on="customer_id",
    how="outer",
    suffixes=("_source", "_target"),
    indicator=True
)

comparison["balance_match"] = (
    comparison["balance_source"] == comparison["balance_target"]
)

comparison["name_match"] = (
    comparison["customer_name_source"] == comparison["customer_name_target"]
)

comparison["kyc_match"] = (
    comparison["kyc_status_source"] == comparison["kyc_status_target"]
)

comparison[
    ["customer_id", "_merge", "name_match", "kyc_match", "balance_match"]
]

,customer_id,_merge,name_match,kyc_match,balance_match
0,101,both,True,True,True
1,102,both,True,True,True
2,103,both,True,True,True
3,104,both,True,True,True
4,107,both,True,True,True
5,108,both,True,True,True
